#### Area Data

##### Importing Libraries and Loading Data

In [1]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
from scipy.spatial import cKDTree

In [2]:
project_root = Path.cwd().parent
raw = project_root/'data'/'raw'
out_dir = project_root/'data'/'processed'/'area'

In [16]:
# British National Grid Code: Coordinates in metres east/north, so distances are real metres
# Every map layer gets converted to this so they line up with each other

bng = 'EPSG:27700'

In [4]:
# Column Mapping

cols = {
    'Postcode': 'postcode', 'In Use?': 'in_use',
    'Latitude': 'lat', 'Longitude': 'lon', 'Easting': 'easting', 'Northing': 'northing',
    'LSOA21 Code': 'lsoa_code', 'Ward Code': 'ward_code', 'Ward': 'ward_name',
    'District Code': 'borough_code', 'District': 'borough_name',
}

In [5]:
# Loading the London Postcode file we have been using before

pc = pd.read_csv(project_root/'data'/'supplementary'/'london_postcodes.csv', usecols=list(cols))

In [6]:
# Change column names

pc = pc.rename(columns=cols)

In [7]:
print(pc.shape)
pc['in_use'].value_counts()

(332308, 11)


in_use
Yes    180983
No     151324
Name: count, dtype: int64

##### Keeping Live Postcodes and Column Check

In [8]:
pc = pc[pc['in_use'] == 'Yes']
pc = pc[['postcode', 'lat', 'lon', 'easting', 'northing',
         'lsoa_code', 'ward_code', 'ward_name', 'borough_code', 'borough_name']].reset_index(drop=True)

In [11]:
print(pc.shape)
print('Missing values:', pc.isna().sum().sum())
print('Postcodes unique:', pc['postcode'].is_unique)

(180982, 10)
Missing values: 0
Postcodes unique: True


In [10]:
pc = pc.dropna()

##### Postcodes Match Check and Map Points

In [13]:
# Ward boundaries (Dec 2024) and the ONS LSOA to ward to borough lookup
# Both are the reference versions every other dataset in this project uses

wards = gpd.read_file(raw/'geography'/'boundaries_wards_2024.geojson')
lookup = pd.read_csv(raw/'geography'/'lookup_lsoa2021_ward2024_borough.csv')

In [14]:
# The one LSOA miss is IG9 6FB, a Redbridge postcode whose LSOA sits over the border in Epping Forest

print('Boroughs:', pc['borough_code'].nunique())
print('Wards:', pc['ward_code'].nunique())
print('Ward codes found in 2024 boundaries:', pc['ward_code'].isin(wards['WD24CD']).mean())
print('LSOA codes found in lookup:', pc['lsoa_code'].isin(lookup['LSOA21CD']).mean())
print(pc.loc[~pc['lsoa_code'].isin(lookup['LSOA21CD']), ['postcode', 'lsoa_code', 'borough_name']])

Boroughs: 33
Wards: 704
Ward codes found in 2024 boundaries: 1.0
LSOA codes found in lookup: 0.9999944745886331
      postcode  lsoa_code borough_name
57098  IG9 6FB  E01021744    Redbridge


In [17]:
# Turn each postcode into a map point using its easting/northing (already in British National Grid metres)
# Only the postcode column is kept, because that's all the spatial join results need

points = gpd.GeoDataFrame(
    pc[['postcode']],
    geometry=gpd.points_from_xy(pc['easting'], pc['northing']),
    crs=bng
)

In [18]:
# Helper function for Zones
# The flood, conservation and article 4 files all need the same three steps
# 1. Load and convert to British National Grid so the shapes line up with the postcode points
# 2. Keep designations still in force (a cancelled one has an end date; empty dates load as NaN or '')
# 3. Repair broken shapes (e.g. outlines that cross themselves), which can break spatial joins

def load_zones(name):
    zones = gpd.read_file(raw/'planning'/f'{name}.geojson').to_crs(bng)
    zones = zones[zones['end-date'].isna() | (zones['end-date'] == '')]
    return zones.set_geometry(zones.geometry.make_valid())

##### Flood Zone

In [19]:
# Environment Agency flood zones 2 (medium probability) and 3 (high probability)

flood = load_zones('planning_flood_risk_zones')

In [20]:
print(flood['flood-risk-level'].value_counts())

flood-risk-level
2    12823
3     4955
Name: count, dtype: int64


In [21]:
# Spatial join: For each postcode point, find every flood shape it falls within (one row per match)
# Zone 3 usually sits inside zone 2, so a postcode can match both and keep the worst (highest)

hits = gpd.sjoin(points, flood[['flood-risk-level', 'geometry']], predicate='within')
worst = hits.groupby('postcode')['flood-risk-level'].max()

In [22]:
# Attach each postcode's worst zone. No match means outside zones 2 and 3,
# which the Environment Agency calls flood zone 1 (low probability)

pc['flood_zone'] = pc['postcode'].map(worst).fillna(1).astype(int)

In [23]:
print(pc['flood_zone'].value_counts())

flood_zone
1    155769
3     20083
2      5130
Name: count, dtype: int64


##### Conservation Areas and Article 4

In [24]:
# Conservation areas: places of special historic character where extensions,
# windows, demolition and tree work need extra permission. Matters to buyers planning works

conservation = load_zones('planning_conservation_areas')

In [25]:
# Spatial join: which conservation area each postcode sits inside

hits = gpd.sjoin(points, conservation[['name', 'geometry']], predicate='within')

In [26]:
# Attach the area's name

pc['conservation_area'] = pc['postcode'].map(hits.groupby('postcode')['name'].first())

In [27]:
# Article 4 directions: areas where the council has removed permitted development rights,
# e.g. converting a house to flats or an office to homes needs full planning permission

article4 = load_zones('planning_article4_areas')

In [28]:
# Spatial Join

hits = gpd.sjoin(points, article4[['name', 'geometry']], predicate='within')

In [29]:
# A postcode can be covered by several directions, so keep them all as one readable string

pc['article4'] = pc['postcode'].map(hits.groupby('postcode')['name'].agg(lambda s: '; '.join(sorted(set(s)))))

In [31]:
print('In a Conservation Area:', round(pc['conservation_area'].notna().mean() * 100, 1), '%')
print('Covered by Article 4 direction:', round(pc['article4'].notna().mean() * 100, 1), '%')

In a Conservation Area: 25.1 %
Covered by Article 4 direction: 43.7 %


##### Transport Acces (PTAL)

In [32]:
# TfL's PTAL (Public Transport Accessibility Level) on a 100 m grid of points across London
# PTAL bands run 0 (worst) to 6b (best); AI (Access Index) is the continuous score behind the band

ptal = pd.read_csv(raw/'transport'/'transport_ptal_100m_grid_2023.csv')

In [33]:
# The grid points come as lat/lon (EPSG:4326); convert to British National Grid to match the postcodes

grid = gpd.points_from_xy(ptal['lon'], ptal['lat'], crs='EPSG:4326').to_crs(bng)

In [34]:
# Build a KD-tree from the grid's (easting, northing) pairs: a structure that finds
# the nearest point in microseconds instead of checking all 159k points

tree = cKDTree(list(zip(grid.x, grid.y)))

In [35]:
# For every postcode at once: dist = metres to the nearest grid point, idx = that point's row number

dist, idx = tree.query(pc[['easting', 'northing']].to_numpy())

In [36]:
#  Use those row numbers to pick each postcode's PTAL band and Access Index

pc['ptal'] = ptal['PTAL_2023'].to_numpy()[idx]
pc['ptal_ai'] = ptal['AI'].to_numpy()[idx].round(1)

In [38]:
print('Distance to nearest grid point (Meters):', round(dist.mean(), 1), 'mean,', round(dist.max(), 1), 'max')
print(pc['ptal'].value_counts().sort_index())

Distance to nearest grid point (Meters): 38.0 mean, 78.9 max
ptal
0      3393
1a     5693
1b    22269
2     44831
3     30945
4     21025
5     13265
6a    18844
6b    20717
Name: count, dtype: int64


##### Air Quality

In [39]:
# Modelled 2022 annual mean NO2, PM2.5 and PM10 (µg/m³) from the London Atmospheric Emissions

air = pd.read_csv(raw/'air'/'air_quality_no2_pm25_pm10_100m_grid_2022.csv')

In [40]:
# Rename columns to avoid clash

air = air.rename(columns={'easting': 'cell_e', 'northing': 'cell_n', 'pm10m': 'pm10'})

In [41]:
# Work out which 100m square each postcode falls in: round down to the square's edge, add 50 for its centre

pc['cell_e'] = pc['easting'] // 100 * 100 + 50
pc['cell_n'] = pc['northing'] // 100 * 100 + 50
pc = pc.merge(air, on=['cell_e', 'cell_n'], how='left').drop(columns=['cell_e', 'cell_n'])

In [42]:
print('Missing air values:', pc[['no2', 'pm25', 'pm10']].isna().sum().sum())
print(pc[['no2', 'pm25', 'pm10']].describe().round(1))

Missing air values: 0
            no2      pm25      pm10
count  180982.0  180982.0  180982.0
mean       22.7       9.4      16.0
std         4.2       0.8       2.1
min        12.4       7.1      11.4
25%        19.8       8.8      14.3
50%        22.1       9.3      15.6
75%        25.3       9.9      17.2
max        51.4      16.4      30.0


In [44]:
pc.shape

(180982, 18)

In [43]:
pc.head()

,postcode,lat,lon,easting,northing,lsoa_code,ward_code,ward_name,borough_code,borough_name,flood_zone,conservation_area,article4,ptal,ptal_ai,no2,pm25,pm10
0,BR1 1AA,51.401545,0.015440,540293.0,168873.0,E01034386,E05013991,Bromley Town,E09000006,Bromley,1,NaN,NaN,6a,29.5,17.943,8.570,13.907
1,BR1 1AB,51.406333,0.015234,540264.0,169405.0,E01000676,E05013991,Bromley Town,E09000006,Bromley,1,Bromley Town Centre,NaN,6a,35.0,20.568,9.006,15.030
2,BR1 1AE,51.404543,0.014221,540199.0,169204.0,E01000677,E05013991,Bromley Town,E09000006,Bromley,1,Bromley Town Centre,NaN,6a,31.4,18.899,8.750,14.231
3,BR1 1AF,51.401391,0.014973,540261.0,168855.0,E01034386,E05013991,Bromley Town,E09000006,Bromley,1,NaN,NaN,6a,29.5,17.943,8.570,13.907
4,BR1 1AG,51.401391,0.014973,540261.0,168855.0,E01034386,E05013991,Bromley Town,E09000006,Bromley,1,NaN,NaN,6a,29.5,17.943,8.570,13.907


In [45]:
# Saving
pc.to_parquet(out_dir/'postcodes.parquet', index=False)

#### Ward Profile

In [46]:
# London ONS Code
london = 'E12000007'

In [47]:
# Start the ward profile from the official 2024 ward list: code, name, borough code, borough name

profile = wards[['WD24CD', 'WD24NM', 'LAD24CD', 'LAD24NM']].copy()

In [48]:
# Renaming Columns

profile.columns = ['ward_code', 'ward_name', 'borough_code', 'borough_name']

In [49]:
# Add one extra row for London as a whole so it will hold the London averages,
# then make ward_code the row label so later tables line up by code automatically

profile.loc[len(profile)] = [london, 'London', london, 'London']
profile = profile.set_index('ward_code')

In [50]:
print(profile.shape)
print(profile.tail(2))

(705, 3)
              ward_name borough_code borough_name
ward_code                                        
E05014119  West Dulwich    E09000022      Lambeth
E12000007        London    E12000007       London


In [51]:
# Helper Function adds LSOA level counts up to wards. Census and population data come by LSOA,
# and the ONS lookup assigns each LSOA to one ward ("best fit"). We sum COUNTS here and
# turn them into percentages afterwards. Averaging LSOA percentages would weight areas wrongly
# The last step adds a London row = the total over all wards

def to_ward(df, lsoa_col, value_cols):
    df = df.merge(lookup[['LSOA21CD', 'WD24CD']], left_on=lsoa_col, right_on='LSOA21CD')
    ward = df.groupby('WD24CD')[value_cols].sum()
    ward.loc[london] = ward.sum()
    return ward

##### Population

In [52]:
# English Indices of Deprivation 2025

iod = pd.read_csv(raw/'deprivation'/'deprivation_iod2025_lsoa.csv')

In [53]:
# Add LSOA populations up to wards, plus the London tota

pop = to_ward(iod, 'LSOA code (2021)', ['Total population: mid 2022'])

In [54]:
# Attach population to the profile. Rows match by ward code and wards with no LSOA get NaN

profile['population'] = pop['Total population: mid 2022']

In [58]:
print('London population:', int(profile.loc[london, 'population']))
print('Wards without population:', profile['population'].isna().sum())
print(profile.loc[profile['population'].isna(), 'borough_name'].unique())

London population: 8869043
Wards without population: 20
<ArrowStringArray>
['City of London']
Length: 1, dtype: str


##### Crime

In [59]:
# Met Police recorded crime. One row per ward x crime subgroup, one column per month (Oct 2024 - Sep 2026)

crime = pd.read_csv(raw/'crime'/'crime_ward_monthly_2024_10_to_2026_09.csv')

In [60]:
print(crime.shape)

(18820, 29)


In [61]:
# Keep rows that are:
# 1. one of our 2024 wards: drops the 17 old "(NOT ACTIVE)" wards and the "NON ATHENA FORCE" row in one go
# 2. not fraud: fraud reports go to Action Fraud and are recorded where the report was taken,
#    not where the crime happened, so they say nothing about the area

crime = crime[crime['WardCode'].isin(profile.index) & ~crime['Group'].isin(['NFIB FRAUD', 'FRAUD AND FORGERY'])]

In [62]:
print(crime.shape)
print('Wards with crime data:', crime['WardCode'].nunique())

(18600, 29)
Wards with crime data: 679


In [63]:
# Month columns are the ones whose names are all digits (e.g. '202510'). YYYYMM text sorts like dates,
# so the last 12 are the latest year

months = sorted(c for c in crime.columns if c.isdigit())[-12:]
print('Last 12 months:', months[0], 'to', months[-1])

Last 12 months: 202510 to 202609


In [64]:
# Add the 12 months across each row into one column

crime = crime.assign(crimes_12m=crime[months].sum(axis=1))

In [65]:
# Reshape Long and Wide

counts = crime.pivot_table(index='WardCode', columns='Group', values='crimes_12m', aggfunc='sum', fill_value=0)

In [66]:
# Cleaning Column Names

counts.columns = 'crime_' + counts.columns.str.lower().str.replace(' ', '_')
counts['crime_total'] = counts.sum(axis=1)

In [67]:
print(counts.shape)
print(counts.sum().sort_values(ascending=False))

(679, 12)
Group
crime_total                                   878626
crime_theft                                   263626
crime_violence_against_the_person             254237
crime_vehicle_offences                         78394
crime_public_order_offences                    57236
crime_arson_and_criminal_damage                51443
crime_drug_offences                            48435
crime_burglary                                 45783
crime_robbery                                  30804
crime_sexual_offences                          27463
crime_miscellaneous_crimes_against_society     13472
crime_possession_of_weapons                     7733
dtype: int64


In [68]:
# Population of exactly the 679 wards that have crime data

met_pop = profile.loc[counts.index, 'population']

In [69]:
print('Missing population in Met wards:', met_pop.isna().sum())

Missing population in Met wards: 0


In [70]:
# Crimes per 1,000 residents per year, so wards of different sizes compare fairly

rates = counts.div(met_pop, axis=0) * 1000
rates.loc[london] = counts.sum() / met_pop.sum() * 1000

In [71]:
# Attaching rates to profile

profile = profile.join(rates.round(2))
print(profile.loc[london, rates.columns].sort_values(ascending=False))

Group
crime_total                                   99.19
crime_theft                                   29.76
crime_violence_against_the_person             28.70
crime_vehicle_offences                         8.85
crime_public_order_offences                    6.46
crime_arson_and_criminal_damage                5.81
crime_drug_offences                            5.47
crime_burglary                                 5.17
crime_robbery                                  3.48
crime_sexual_offences                          3.10
crime_miscellaneous_crimes_against_society     1.52
crime_possession_of_weapons                    0.87
Name: E12000007, dtype: float64


##### Age Mix

In [72]:
# Census 2021 table TS007A: residents per LSOA in eighteen 5-year age bands

age = pd.read_csv(raw/'census'/'census2021_age_bands_lsoa.csv')

In [73]:
# The 18 band columns in file order (0-4, 5-9, ... 85+)

age_cols = [c for c in age.columns if c.startswith('Age: Aged')]

In [74]:
# Add age counts upto wards and London

age_ward = to_ward(age, 'geography code', ['Age: Total'] + age_cols)

In [75]:
# Group the 18 bands into five life stages by position
# 30-44 is the typical first-time buyer and young family age

bands = {
    'pct_age_0_14': age_cols[0:3],
    'pct_age_15_29': age_cols[3:6],
    'pct_age_30_44': age_cols[6:9],
    'pct_age_45_64': age_cols[9:13],
    'pct_age_65_plus': age_cols[13:],
}

In [76]:
for name, band_cols in bands.items():
    print(name, '=', band_cols[0], '...', band_cols[-1])
    profile[name] = (age_ward[band_cols].sum(axis=1) / age_ward['Age: Total'] * 100).round(1)

pct_age_0_14 = Age: Aged 4 years and under ... Age: Aged 10 to 14 years
pct_age_15_29 = Age: Aged 15 to 19 years ... Age: Aged 25 to 29 years
pct_age_30_44 = Age: Aged 30 to 34 years ... Age: Aged 40 to 44 years
pct_age_45_64 = Age: Aged 45 to 49 years ... Age: Aged 60 to 64 years
pct_age_65_plus = Age: Aged 65 to 69 years ... Age: Aged 85 years and over


##### Housing Tenure

In [77]:
# Census 2021 table TS054: households per LSOA by how they occupy their home

tenure = pd.read_csv(raw/'census'/'census2021_housing_tenure_lsoa.csv')

In [78]:
# Every column name starts with the same text, so store it once and build the names from it
# Owned = outright + mortgage. Shared ownership and rent-free are left out

prefix = 'Tenure of household: '
total = prefix + 'Total: All households'
tenure_cols = {'Owned': 'pct_owned', 'Social rented': 'pct_social_rented', 'Private rented': 'pct_private_rented'}

In [79]:
# Add household counts up to wards and London (total first, then the three tenures)

tenure_ward = to_ward(tenure, 'geography code', [total] + [prefix + c for c in tenure_cols])

In [80]:
# Keep the household count to get a sense of the ward's size, then each tenure as a percent of households

profile['households'] = tenure_ward[total]
for col, name in tenure_cols.items():
    profile[name] = (tenure_ward[prefix + col] / tenure_ward[total] * 100).round(1)

In [81]:
print(profile.shape)
print('Age shares add to 100:', profile[list(bands)].dropna().sum(axis=1).between(99.5, 100.5).all())
print(profile.loc[london])

(705, 25)
Age shares add to 100: True
ward_name                                        London
borough_code                                  E12000007
borough_name                                     London
population                                    8869043.0
crime_arson_and_criminal_damage                    5.81
crime_burglary                                     5.17
crime_drug_offences                                5.47
crime_miscellaneous_crimes_against_society         1.52
crime_possession_of_weapons                        0.87
crime_public_order_offences                        6.46
crime_robbery                                      3.48
crime_sexual_offences                               3.1
crime_theft                                       29.76
crime_vehicle_offences                             8.85
crime_violence_against_the_person                  28.7
crime_total                                       99.19
pct_age_0_14                                       18.1
pct_age_15

In [86]:
profile.shape

(705, 25)

##### Transport and AIr Quality

In [88]:
# Ward Averages of Transport access and Air Quality

loc_cols = ['ptal_ai', 'no2', 'pm25', 'pm10']
local = pc.groupby('ward_code')[loc_cols].mean()
local.loc[london] = pc[loc_cols].mean()

In [89]:
# Turn the average Access Index back into TfL's PTAL band using TfL's official cut-offs
# right=False means each band includes its lower edge, e.g. 2.5 is band 1b, not 1a

ptal_bins = [0, 0.01, 2.5, 5, 10, 15, 20, 25, 40, float('inf')]
ptal_labels = ['0', '1a', '1b', '2', '3', '4', '5', '6a', '6b']
local['ptal_band'] = pd.cut(local['ptal_ai'], bins=ptal_bins, labels=ptal_labels, right=False).astype(str)

In [90]:
# Attach to the profile (rows match by ward code), rounded for readability
profile = profile.join(local.round({'ptal_ai': 1, 'no2': 1, 'pm25': 1, 'pm10': 1}))
print(profile[['ptal_ai', 'ptal_band', 'no2', 'pm25', 'pm10']].isna().sum().sum())
print(profile.loc[london, ['ptal_ai', 'ptal_band', 'no2', 'pm25', 'pm10']])
print(profile['ptal_band'].value_counts().sort_index())

0
ptal_ai      18.5
ptal_band       4
no2          22.7
pm25          9.4
pm10         16.0
Name: E12000007, dtype: object
ptal_band
1a      3
1b     35
2     253
3     171
4      76
5      72
6a     49
6b     46
Name: count, dtype: int64


##### House Prices

In [91]:
# UK House Price Index (HM Land Registry): monthly average prices by borough, London and England
hpi = pd.read_csv(raw/'house_prices'/'house_price_index_borough_monthly_to_2026_07.csv')

In [92]:
# Dates are day-first text ('01/07/2026'), so convert them to real dates, then keep the latest month
hpi['Date'] = pd.to_datetime(hpi['Date'], dayfirst=True)
latest = hpi[hpi['Date'] == hpi['Date'].max()]
print('Latest month:', latest['Date'].iloc[0].strftime('%Y-%m'), '| areas:', len(latest))

Latest month: 2026-07 | areas: 35


In [93]:
# Average price, its change over 12 months, and the average by property type
prices = latest.set_index('AreaCode')[['AveragePrice', '12m%Change', 'FlatPrice', 'TerracedPrice',
                                        'SemiDetachedPrice', 'DetachedPrice']]
prices.columns = ['price_avg', 'price_change_12m', 'price_flat', 'price_terraced', 'price_semi', 'price_detached']
prices['price_month'] = latest['Date'].iloc[0].strftime('%Y-%m')
print(prices.loc[['E09000012', london]])

           price_avg  price_change_12m  price_flat  price_terraced  \
AreaCode                                                             
E09000012     605729              -2.6    524885.0        960633.0   
E12000007     550037              -3.3    424013.0        642452.0   

           price_semi  price_detached price_month  
AreaCode                                           
E09000012   1193876.0       1330617.0     2026-07  
E12000007    724106.0       1167718.0     2026-07  


##### Rents

In [94]:
# ONS Price Index of Private Rents: monthly average rent by borough, London and England
rents = pd.read_csv(raw/'rents'/'private_rents_borough_monthly_2015_01_to_2026_08.csv')

In [95]:
# 'Time period' is 'YYYY-MM-DD' text, which sorts like dates, so max() is the latest month
latest_rent = rents[rents['Time period'] == rents['Time period'].max()]
print('Latest month:', latest_rent['Time period'].iloc[0][:7], '| areas:', len(latest_rent))

Latest month: 2026-08 | areas: 34


In [96]:
# Average monthly rent, its annual % change, and rent by number of bedrooms
# Annual change is stored as text because old months contain '[x]' (not available),
# so convert every column to numbers; anything that isn't a number becomes NaN
rent = latest_rent.set_index('Area code')[['Rental price', 'Annual change', 'Rental price one bed',
                                           'Rental price two bed', 'Rental price three bed',
                                           'Rental price four or more bed']]
rent = rent.apply(pd.to_numeric, errors='coerce')
rent.columns = ['rent_avg', 'rent_change_12m', 'rent_1bed', 'rent_2bed', 'rent_3bed', 'rent_4bed_plus']
rent['rent_change_12m'] = rent['rent_change_12m'].round(1)
rent['rent_month'] = latest_rent['Time period'].iloc[0][:7]
print(rent.loc[['E09000012', london]])

           rent_avg  rent_change_12m  rent_1bed  rent_2bed  rent_3bed  \
Area code                                                               
E09000012      2658              3.6       2001       2489       2846   
E12000007      2332              3.5       1760       2232       2643   

           rent_4bed_plus rent_month  
Area code                             
E09000012            3674    2026-08  
E12000007            3646    2026-08  


##### Council Tax

In [97]:
# MHCLG Council Tax 2026-27, Table 10: one row per London borough
ct = pd.read_csv(raw/'council_tax'/'council_tax_borough_2026_27.csv')

In [98]:
# Band D bill for the area, including the borough's and the GLA's share (line 17 of the return)
# Tax base (line 7) = the number of Band D-equivalent homes the bill is spread over
band_d = [c for c in ct.columns if c.startswith('17.')][0]
tax_base = '7. Council tax base for council tax setting purposes (current year)'

tax = ct.set_index('ONS Code')[[band_d]].rename(columns={band_d: 'council_tax_band_d'})

# London average = each borough's Band D weighted by its tax base (big boroughs count more),
# the same way the government calculates an area's average Band D
tax.loc[london] = (ct[band_d] * ct[tax_base]).sum() / ct[tax_base].sum()
tax = tax.round(2)
print(tax.loc[['E09000012', 'E09000032', london]])

           council_tax_band_d
ONS Code                     
E09000012             2060.30
E09000032             1028.21
E12000007             2068.47


##### Borough Figures into Wards

In [99]:
# Combine the three borough tables. prices has every borough, so it's the base;
# boroughs missing from rents (City of London) get NaN
borough = prices.join(rent).join(tax)

# Attach borough figures to every ward via its borough_code. The London row has
# borough_code E12000007, so it picks up the London figures automatically
profile = profile.join(borough, on='borough_code')
print(profile.shape)

(705, 45)


In [100]:
# Missing values per column
print(profile.isna().sum()[lambda s: s > 0])
print(profile.loc[london])

population                                    20
crime_arson_and_criminal_damage               25
crime_burglary                                25
crime_drug_offences                           25
crime_miscellaneous_crimes_against_society    25
crime_possession_of_weapons                   25
crime_public_order_offences                   25
crime_robbery                                 25
crime_sexual_offences                         25
crime_theft                                   25
crime_vehicle_offences                        25
crime_violence_against_the_person             25
crime_total                                   25
pct_age_0_14                                  20
pct_age_15_29                                 20
pct_age_30_44                                 20
pct_age_45_64                                 20
pct_age_65_plus                               20
households                                    20
pct_owned                                     20
pct_social_rented   

In [101]:
# Saving

profile.reset_index().to_parquet(out_dir/'ward_profile.parquet', index=False)

In [102]:
# Compare one ward (Barking Riverside) with London side by side

profile.loc[['E05014055', london], ['ward_name', 'ptal_band', 'no2', 'price_avg', 'rent_2bed', 'council_tax_band_d']]

,ward_name,ptal_band,no2,price_avg,rent_2bed,council_tax_band_d
ward_code,,,,,,
E05014055,Barking Riverside,2,19.0,375604,1721.0,2198.51
E12000007,London,4,22.7,550037,2232.0,2068.47
